# Imports

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm

# Config

In [ ]:
config = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "data_directory": "../data",
}

augmentation_config = {
    "size": 32,
    "scale": (0.2, 1.0),
    "p_random_horizontal_flip": 0.5,
    "brightness": 0.4,
    "contrast": 0.4,
    "saturation": 0.4,
    "hue": 0.1,
    "p_color_jitter": 0.8,
    "p_grayscale": 0.2,
    "mean": (0.4914, 0.4822, 0.4465),
    "std": (0.2023, 0.1994, 0.2010),
}

training_config = {
    "epochs": 20,        # Paper uses 500; set higher for real runs.
    "batch_size": 128,   # Paper uses 512.
    "lr": 0.4,
    "momentum": 0.9,
    "weight_decay": 0.0001,
}

cluster_config = {
    "max_clusters": 500,  # Paper uses 500 for CIFAR-10/100 (Appendix F.1).
    "B": 10,              # Budget per round. B = M (10 classes) matches paper.
    "min_cluster_size": 5,  # Drop clusters smaller than this (Appendix F.1).
}

# Reproducibility

In [ ]:
def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(config["seed"])

# Data Augmentation

In [ ]:
class StochasticDataAugmentation:

    def __init__(self, base_transform: transforms.Compose, num_views: int = 2):
        self.base_transform = base_transform
        self.num_views = num_views

    def __call__(self, x):
        return [self.base_transform(x) for _ in range(self.num_views)]


contrastive_base_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        size=augmentation_config["size"],
        scale=augmentation_config["scale"],
    ),
    transforms.RandomHorizontalFlip(p=augmentation_config["p_random_horizontal_flip"]),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=augmentation_config["brightness"],
            contrast=augmentation_config["contrast"],
            saturation=augmentation_config["saturation"],
            hue=augmentation_config["hue"],
        ),
    ], p=augmentation_config["p_color_jitter"]),
    transforms.RandomGrayscale(p=augmentation_config["p_grayscale"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=augmentation_config["mean"], std=augmentation_config["std"]),
])

contrastive_transform = StochasticDataAugmentation(contrastive_base_transform, num_views=2)

embedding_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=augmentation_config["mean"], std=augmentation_config["std"]),
])

# SimCLR Model

In [ ]:
class SimCLRModel(nn.Module):

    def __init__(self, feature_dimension: int = 128) -> None:
        super().__init__()
        self.base_encoder = torchvision.models.resnet18(weights=None)
        in_features = self.base_encoder.fc.in_features
        self.base_encoder.fc = nn.Identity()

        self.mlp = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.ReLU(),
            nn.Linear(in_features, feature_dimension),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mlp(self.base_encoder(x))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            return F.normalize(self.base_encoder(x), p=2, dim=1)


# SimCLR Loss (NT-Xent)

In [ ]:
class SimCLRLoss(nn.Module):

    def __init__(self, temperature: float = 0.5) -> None:
        super().__init__()
        self.temperature = temperature

    def forward(self, z_i: torch.Tensor, z_j: torch.Tensor) -> torch.Tensor:
        N = z_i.size(0)
        z = F.normalize(torch.cat([z_i, z_j], dim=0), p=2, dim=1)  # (2N, d)

        # Cosine similarity matrix scaled by temperature
        sim = torch.matmul(z, z.T) / self.temperature            # (2N, 2N)
        # Mask diagonal (self-similarity) with a large negative value
        sim.masked_fill_(torch.eye(2 * N, device=z.device).bool(), -1e9)

        # Positive for index k is index k+N (and vice-versa)
        labels = (torch.arange(2 * N, device=z.device) + N) % (2 * N)
        return F.cross_entropy(sim, labels)

# Training Helpers

In [ ]:
def similarity_statistics(z_i: torch.Tensor, z_j: torch.Tensor) -> tuple:
    with torch.no_grad():
        N = z_i.size(0)
        z = F.normalize(torch.cat([z_i, z_j], dim=0), dim=1)
        sim = torch.matmul(z, z.T)

        positives = torch.cat([torch.diag(sim, N), torch.diag(sim, -N)])
        mask = torch.eye(2 * N, device=z.device).bool()
        negatives = sim[~mask].view(2 * N, -1)

        return positives.mean().item(), negatives.mean().item()

# SimCLR Training (Step 1 of TPC_RP: Representation Learning)

In [ ]:
device = torch.device(config["device"])
print(f"Using device: {device}")

model = SimCLRModel().to(device)
optimizer = optim.SGD(
    model.parameters(),
    lr=training_config["lr"],
    momentum=training_config["momentum"],
    weight_decay=training_config["weight_decay"],
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=training_config["epochs"],
    eta_min=0,
)
criterion = SimCLRLoss()

train_dataset = torchvision.datasets.CIFAR10(
    root=config["data_directory"], train=True, download=True, transform=contrastive_transform
)
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=training_config["batch_size"],
    shuffle=True,
    num_workers=2,
    persistent_workers=True,
    pin_memory=True,
    drop_last=True,
)

for epoch in range(training_config["epochs"]):
    model.train()
    total_loss = total_pos_sim = total_neg_sim = 0.0

    for views, _ in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{training_config['epochs']}"):
        x_i, x_j = views[0].to(device), views[1].to(device)
        z_i, z_j = model(x_i), model(x_j)

        pos_sim, neg_sim = similarity_statistics(z_i, z_j)
        loss = criterion(z_i, z_j)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_pos_sim += pos_sim
        total_neg_sim += neg_sim

    scheduler.step()
    n = len(train_loader)
    print(
        f"Epoch [{epoch + 1}/{training_config['epochs']}] "
        f"Loss: {total_loss / n:.4f}  "
        f"Positive similarity: {total_pos_sim / n:.3f}  "
        f"Negative similarity: {total_neg_sim / n:.3f}  "
        f"Learning rate: {scheduler.get_last_lr()[0]:.6f}"
    )

torch.save(model.state_dict(), "simclr_model.pth")

# Build Embeddings (L2-Normalised penultimate layer, 512-d)

In [ ]:
model.load_state_dict(torch.load("simclr_model.pth", weights_only=True))
model.eval()

embedding_dataset = torchvision.datasets.CIFAR10(
    root=config["data_directory"], train=True, download=True, transform=embedding_transform
)
embedding_loader = torch.utils.data.DataLoader(
    embedding_dataset,
    batch_size=training_config["batch_size"],
    shuffle=False,
    num_workers=2,
    persistent_workers=True,
    pin_memory=True,
    drop_last=False,
)

all_embeddings_list, all_labels_list = [], []
with torch.no_grad():
    for images, labels in tqdm(embedding_loader, desc="Embedding dataset"):
        all_embeddings_list.append(model.encode(images.to(device)).cpu())
        all_labels_list.append(labels)

all_embeddings = torch.cat(all_embeddings_list).numpy()
all_labels = torch.cat(all_labels_list).numpy()
print(f"Embeddings: {all_embeddings.shape}, Labels: {all_labels.shape}")

# Typicality - Equation 4 in the paper

In [ ]:
def compute_typicality(cluster_embeddings: np.ndarray, K: int = 20) -> np.ndarray:
    cluster_size = cluster_embeddings.shape[0]
    # Need at least 1 neighbour; cap so K+1 <= cluster_size.
    K = min(K, cluster_size - 1)
    if K <= 0:
        # Single-point cluster: trivially most typical.
        return np.array([1.0])

    nn_model = NearestNeighbors(n_neighbors=K + 1, metric="euclidean")
    nn_model.fit(cluster_embeddings)
    distances, _ = nn_model.kneighbors(cluster_embeddings)

    # distances[:, 0] is always 0 (self); skip it.
    mean_dist = np.mean(distances[:, 1:], axis=1)
    return 1.0 / (mean_dist + 1e-8)

# TPC_RP Query Selection  (Steps 2 & 3 of TPC_RP)

In [ ]:
def run_typiclust_round(
    all_embeddings: np.ndarray,
    labeled_set: set,
    B: int,
    max_clusters: int = 500,
    min_cluster_size: int = 5,
) -> list:
    K = min(len(labeled_set) + B, max_clusters)
    clustering_model = (
        KMeans(n_clusters=K, n_init=10)
        if K <= 50
        else MiniBatchKMeans(n_clusters=K, batch_size=1024, n_init=10)
    )
    cluster_labels = clustering_model.fit_predict(all_embeddings)

    # Clusters that already contain at least one labeled example.
    labeled_cluster_ids = set(cluster_labels[i] for i in labeled_set)

    new_indices = []
    remaining = B
    temp_labels = cluster_labels.copy()

    while remaining > 0:
        # Uncovered clusters large enough for reliable typicality estimation.
        eligible = [
            c for c in range(K)
            if c not in labeled_cluster_ids
            and np.sum(temp_labels == c) >= min_cluster_size  # FIX: >= 5, not > 5
        ]
        if not eligible:
            # Relax the 'uncovered' constraint as a fallback (per Appendix F.1).
            eligible = [
                c for c in range(K)
                if np.sum(temp_labels == c) >= min_cluster_size
            ]
        if not eligible:
            break

        # Pick the largest eligible cluster (Appendix F.1, Step 3).
        selected_cluster_id = max(eligible, key=lambda c: np.sum(temp_labels == c))

        cluster_indices = np.where(temp_labels == selected_cluster_id)[0]
        cluster_embeddings = all_embeddings[cluster_indices]

        typicality_scores = compute_typicality(cluster_embeddings, K=20)
        best_local_idx = np.argmax(typicality_scores)
        chosen_idx = int(cluster_indices[best_local_idx])

        new_indices.append(chosen_idx)
        labeled_cluster_ids.add(selected_cluster_id)
        # Mark entire cluster as consumed so it is not selected again.
        temp_labels[temp_labels == selected_cluster_id] = -1
        remaining -= 1

    return new_indices

# Softmax helper (shared by Uncertainty, Margin, Entropy, DBAL, BALD)

In [ ]:
def get_softmax_scores(
    model: nn.Module,
    dataset: torch.utils.data.Dataset,
    indices: list,
    device: torch.device,
    n_dropout: int = 10,
    use_dropout: bool = False,
) -> np.ndarray:
    subset = torch.utils.data.Subset(dataset, indices)
    loader = torch.utils.data.DataLoader(
        subset, batch_size=256, shuffle=False, num_workers=2
    )

    if use_dropout:
        model.train()  # Activate dropout layers.
        all_passes = []
        for _ in range(n_dropout):
            probs = []
            with torch.no_grad():
                for images, _ in loader:
                    probs.append(
                        F.softmax(model(images.to(device)), dim=1).cpu().numpy()
                    )
            all_passes.append(np.concatenate(probs, axis=0))
        model.eval()
        return np.stack(all_passes, axis=0)  # (n_dropout, n_samples, n_classes)
    else:
        model.eval()
        probs = []
        with torch.no_grad():
            for images, _ in loader:
                probs.append(
                    F.softmax(model(images.to(device)), dim=1).cpu().numpy()
                )
        return np.concatenate(probs, axis=0)  # (n_samples, n_classes)

# Baseline AL Strategies

## 1. Random

In [ ]:
def run_random_round(labeled_set: set, B: int, total: int = 50000) -> list:
    pool = list(set(range(total)) - labeled_set)
    return random.sample(pool, B)

## 2. Uncertainy - Lowest max softmax (least confident)

In [ ]:
def run_uncertainty_round(
    model: nn.Module,
    dataset: torch.utils.data.Dataset,
    labeled_set: set,
    B: int,
    device: torch.device,
) -> list:
    pool = list(set(range(len(dataset))) - labeled_set)
    probs = get_softmax_scores(model, dataset, pool, device)
    max_probs = probs.max(axis=1)
    chosen_local = np.argsort(max_probs)[:B]  # Smallest confidence = most uncertain.
    return [pool[i] for i in chosen_local]

## 3. Margin — smallest gap between top-2 softmax outputs

In [ ]:
def run_margin_round(
    model: nn.Module,
    dataset: torch.utils.data.Dataset,
    labeled_set: set,
    B: int,
    device: torch.device,
) -> list:
    pool = list(set(range(len(dataset))) - labeled_set)
    probs = get_softmax_scores(model, dataset, pool, device)
    sorted_probs = np.sort(probs, axis=1)[:, ::-1]       # Descending.
    margins = sorted_probs[:, 0] - sorted_probs[:, 1]    # Top-1 minus top-2.
    chosen_local = np.argsort(margins)[:B]                # Smallest margin first.
    return [pool[i] for i in chosen_local]

## 4. Entropy — highest predictive entropy

In [ ]:
def run_entropy_round(
    model: nn.Module,
    dataset: torch.utils.data.Dataset,
    labeled_set: set,
    B: int,
    device: torch.device,
) -> list:
    pool = list(set(range(len(dataset))) - labeled_set)
    probs = get_softmax_scores(model, dataset, pool, device)
    entropy = -np.sum(probs * np.log(probs + 1e-8), axis=1)
    chosen_local = np.argsort(entropy)[::-1][:B]          # Highest entropy first.
    return [pool[i] for i in chosen_local]

## 5. DBAL — Deep Bayesian Active Learning (Gal et al. 2017)
MC-Dropout; queries points with highest predictive entropy.


In [ ]:
def add_dropout_to_resnet(model: nn.Module, p: float = 0.5) -> nn.Module:
    """Insert a Dropout layer before the final FC layer."""
    model.fc = nn.Sequential(nn.Dropout(p=p), model.fc)
    return model


def run_dbal_round(
    model: nn.Module,
    dataset: torch.utils.data.Dataset,
    labeled_set: set,
    B: int,
    device: torch.device,
    n_dropout: int = 10,
) -> list:
    pool = list(set(range(len(dataset))) - labeled_set)
    # mc_probs: (n_dropout, n_samples, n_classes)
    mc_probs = get_softmax_scores(
        model, dataset, pool, device, n_dropout=n_dropout, use_dropout=True
    )
    mean_probs = mc_probs.mean(axis=0)
    entropy = -np.sum(mean_probs * np.log(mean_probs + 1e-8), axis=1)
    chosen_local = np.argsort(entropy)[::-1][:B]
    return [pool[i] for i in chosen_local]

## 6. CoreSet (Sener & Savarese 2018)
Greedy furthest-first traversal in embedding space.

In [ ]:
def run_coreset_round(
    all_embeddings: np.ndarray,
    labeled_set: set,
    B: int,
) -> list:
    pool_indices = np.array(list(set(range(len(all_embeddings))) - labeled_set))
    pool_embeddings = all_embeddings[pool_indices]

    if labeled_set:
        labeled_embeddings = all_embeddings[np.array(list(labeled_set))]
        # Distance from each pool point to its nearest labeled point.
        dists = pairwise_distances(pool_embeddings, labeled_embeddings, metric="euclidean")
        min_dists = dists.min(axis=1)
    else:
        # Cold start: seed with one random point; all others are infinitely far.
        seed = np.random.randint(len(pool_indices))
        min_dists = np.full(len(pool_indices), np.inf)
        min_dists[seed] = 0.0

    chosen_local = []
    for _ in range(B):
        chosen = int(np.argmax(min_dists))
        chosen_local.append(chosen)
        # Update distances using the newly chosen point.
        new_dists = np.linalg.norm(pool_embeddings - pool_embeddings[chosen], axis=1)
        min_dists = np.minimum(min_dists, new_dists)
        min_dists[chosen] = -np.inf  # Prevent re-selection.

    return [pool_indices[i] for i in chosen_local]

## 7. BALD — Bayesian Active Learning by Disagreement (Kirsch et al. 2019)
Maximises mutual information: $I(y; w | x) = H(y|x) - E_w[H(y|x,w)]$


In [ ]:
def run_bald_round(
    model: nn.Module,
    dataset: torch.utils.data.Dataset,
    labeled_set: set,
    B: int,
    device: torch.device,
    n_dropout: int = 10,
) -> list:
    pool = list(set(range(len(dataset))) - labeled_set)
    # mc_probs: (n_dropout, n_samples, n_classes)
    mc_probs = get_softmax_scores(
        model, dataset, pool, device, n_dropout=n_dropout, use_dropout=True
    )
    mean_probs = mc_probs.mean(axis=0)

    H = -np.sum(mean_probs * np.log(mean_probs + 1e-8), axis=1)       # H(y|x)
    E_H = -np.mean(
        np.sum(mc_probs * np.log(mc_probs + 1e-8), axis=2), axis=0
    )                                                                    # E_w[H(y|x,w)]

    bald_scores = H - E_H
    chosen_local = np.argsort(bald_scores)[::-1][:B]
    return [pool[i] for i in chosen_local]

 ## 8. BADGE — Batch Active learning by Diverse Gradient Embeddings (Ash et al. 2020)
 Gradient of CE loss w.r.t. last-layer weights, then k-means++.

In [ ]:
def get_gradient_embeddings(
    model: nn.Module,
    dataset: torch.utils.data.Dataset,
    labeled_set: set,
    device: torch.device,
) -> tuple:
    pool = list(set(range(len(dataset))) - labeled_set)
    subset = torch.utils.data.Subset(dataset, pool)
    loader = torch.utils.data.DataLoader(
        subset, batch_size=256, shuffle=False, num_workers=2
    )

    model.eval()
    grad_embeddings = []

    for images, _ in loader:
        images = images.to(device)
        with torch.no_grad():
            # Extract penultimate features by chaining ResNet-18 layers.
            h = model.conv1(images)
            h = model.bn1(h)
            h = model.relu(h)
            h = model.maxpool(h)
            h = model.layer1(h)
            h = model.layer2(h)
            h = model.layer3(h)
            h = model.layer4(h)
            features = model.avgpool(h).squeeze(-1).squeeze(-1)   # (B, 512)

            logits = model.fc(features)                           # (B, n_classes)
            probs = F.softmax(logits, dim=1)
            pseudo_labels = probs.argmax(dim=1)

        n_classes = probs.size(1)
        one_hot = F.one_hot(pseudo_labels, n_classes).float()
        grad = (probs - one_hot).cpu().numpy()                    # (B, n_classes)
        feat = features.detach().cpu().numpy()                    # (B, feat_dim)

        # Outer product flattened: (B, n_classes * feat_dim)
        emb = grad[:, :, None] * feat[:, None, :]                # (B, n_classes, feat_dim)
        grad_embeddings.append(emb.reshape(len(images), -1))

    return pool, np.concatenate(grad_embeddings, axis=0)


def kmeans_plus_plus_init(embeddings: np.ndarray, B: int) -> list:
    n = len(embeddings)
    chosen = [np.random.randint(n)]
    for _ in range(B - 1):
        # Distance from each point to its nearest already-chosen centre.
        dists = np.min(
            np.linalg.norm(embeddings[chosen][:, None] - embeddings[None], axis=2) ** 2,
            axis=0,
        )
        dists[chosen] = 0.0
        probs = dists / dists.sum()
        chosen.append(int(np.random.choice(n, p=probs)))
    return chosen


def run_badge_round(
    model: nn.Module,
    dataset: torch.utils.data.Dataset,
    labeled_set: set,
    B: int,
    device: torch.device,
) -> list:
    pool, grad_embs = get_gradient_embeddings(model, dataset, labeled_set, device)
    chosen_local = kmeans_plus_plus_init(grad_embs, B)
    return [pool[i] for i in chosen_local]


# Classifer Training & Evaluation

In [ ]:
def train_and_evaluate(
    labeled_indices: list,
    device: torch.device,
    train_dataset: torch.utils.data.Dataset,
    test_loader: torch.utils.data.DataLoader,
    epochs: int = 100,
    model_override: nn.Module = None,
) -> float:
    if model_override is not None:
        classifier = model_override
    else:
        classifier = torchvision.models.resnet18(weights=None)
        classifier.fc = nn.Linear(classifier.fc.in_features, 10)
        # Kaiming / constant initialisation (Appendix F.2).
        for m in classifier.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
        classifier = classifier.to(device)

    subset = torch.utils.data.Subset(train_dataset, labeled_indices)
    loader = torch.utils.data.DataLoader(
        subset,
        batch_size=min(64, len(labeled_indices)),
        shuffle=True,
        num_workers=2,
        drop_last=False,
    )

    # Appendix F.2: SGD with Nesterov, cosine LR, lr=0.025.
    opt = optim.SGD(
        classifier.parameters(), lr=0.025, momentum=0.9,
        weight_decay=5e-4, nesterov=True,
    )
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    classifier.train()
    for _ in range(epochs):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            opt.zero_grad()
            F.cross_entropy(classifier(images), labels).backward()
            opt.step()
        sched.step()

    classifier.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            correct += (classifier(images).argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct / total

# Active Learning Loop

In [ ]:
test_dataset = torchvision.datasets.CIFAR10(
    root=config["data_directory"], train=False, download=True, transform=embedding_transform
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=256, shuffle=False, num_workers=2
)

full_train_dataset = torchvision.datasets.CIFAR10(
    root=config["data_directory"], train=True, download=True, transform=embedding_transform
)

num_rounds = 5
B = cluster_config["B"]

strategies = {
    "TPC_RP":      {"labeled": set(), "accs": []},
    "Random":      {"labeled": set(), "accs": []},
    "Uncertainty": {"labeled": set(), "accs": []},
    "Margin":      {"labeled": set(), "accs": []},
    "Entropy":     {"labeled": set(), "accs": []},
    "DBAL":        {"labeled": set(), "accs": []},
    "CoreSet":     {"labeled": set(), "accs": []},
    "BALD":        {"labeled": set(), "accs": []},
    "BADGE":       {"labeled": set(), "accs": []},
}

for round_idx in range(num_rounds):
    print(f"\n=== Round {round_idx + 1}/{num_rounds} (B={B}) ===")

    for name, s in strategies.items():
        labeled = s["labeled"]

        # ---- Query selection ------------------------------------------------
        if name == "TPC_RP":
            new_idx = run_typiclust_round(
                all_embeddings, labeled, B,
                max_clusters=cluster_config["max_clusters"],
                min_cluster_size=cluster_config["min_cluster_size"],
            )

        elif name == "Random":
            new_idx = run_random_round(labeled, B, total=len(full_train_dataset))

        elif name == "CoreSet":
            # CoreSet operates purely on embeddings; no classifier needed.
            new_idx = run_coreset_round(all_embeddings, labeled, B)

        elif name in ("Uncertainty", "Margin", "Entropy", "DBAL", "BALD", "BADGE"):
            if len(labeled) == 0:
                # Cold start: no model available → fall back to random.
                new_idx = run_random_round(labeled, B, total=len(full_train_dataset))
            else:
                # Build and train a fresh classifier for this strategy's query.
                strat_model = torchvision.models.resnet18(weights=None)
                strat_model.fc = nn.Linear(strat_model.fc.in_features, 10)
                if name in ("DBAL", "BALD"):
                    strat_model = add_dropout_to_resnet(strat_model)
                strat_model = strat_model.to(device)

                train_and_evaluate(
                    list(labeled), device,
                    full_train_dataset, test_loader,
                    epochs=100, model_override=strat_model,
                )

                if name == "Uncertainty":
                    new_idx = run_uncertainty_round(strat_model, full_train_dataset, labeled, B, device)
                elif name == "Margin":
                    new_idx = run_margin_round(strat_model, full_train_dataset, labeled, B, device)
                elif name == "Entropy":
                    new_idx = run_entropy_round(strat_model, full_train_dataset, labeled, B, device)
                elif name == "DBAL":
                    new_idx = run_dbal_round(strat_model, full_train_dataset, labeled, B, device)
                elif name == "BALD":
                    new_idx = run_bald_round(strat_model, full_train_dataset, labeled, B, device)
                elif name == "BADGE":
                    new_idx = run_badge_round(strat_model, full_train_dataset, labeled, B, device)

        # ---- Update labeled set & evaluate with a fresh model ---------------
        s["labeled"].update(new_idx)
        acc = train_and_evaluate(
            list(s["labeled"]), device, full_train_dataset, test_loader
        )
        s["accs"].append(acc)
        print(f"  {name:12s} | budget={len(s['labeled']):4d} | acc={acc:.4f}")

# Plot Results

In [ ]:
budgets = [B * (i + 1) for i in range(num_rounds)]

plt.figure(figsize=(10, 6))
for name, s in strategies.items():
    plt.plot(budgets, [a * 100 for a in s["accs"]], marker="o", label=name)
plt.xlabel("Cumulative Budget")
plt.ylabel("Test Accuracy (%)")
plt.title("CIFAR-10: Low Budget Active Learning (TPC_RP vs Baselines)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("al_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved al_results.png")